# EternityQuant — Colab ML 训练

在 Colab 的免费 T4 GPU 上训练 EternityQuant 的量化选股模型。

## 支持算法
- **LightGBM** (CPU) — 基线模型，qlib Alpha158 特征
- **MLP** (CUDA) — 自写全连接网络，158→512→256→128→1
- **GRU** (CUDA) — 自写时序模型，6×26 时序重塑，量化选股最佳
- **LSTM** (CUDA) — 与 GRU 同架构，3 门替代 2 门
- **DeepLOB** (CUDA) — CNN+BiLSTM+Attention，顶会论文复现
- **TFT** (CUDA) — Temporal Fusion Transformer，Google 多时间跨度预测

## 多卡 GPU 支持
Colab 单台只有 1 张 T4，但代码支持 `--gpus "0,1,2,3"` 参数，
在租用多 GPU 云服务器（如 A100×4、4090×2）时自动启用 `nn.DataParallel` 并行训练。

## 输出
- 训练好的模型文件（`.pkl`）→ 下载回本地，用 `eq ml register/activate` 导入
- 训练指标（IC、ICIR、Rank IC）

---

## 1. 环境准备

安装依赖 + 克隆 EternityQuant 代码。

In [ ]:
# @title 安装依赖
import sys, os, subprocess

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(pkgs))

# 基础依赖
_pip("numpy", "pandas", "typer", "python-dotenv")

# qlib（Colab 编译时间约 2 分钟）
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "pyqlib>=0.9", "--no-build-isolation"])

# LightGBM
_pip("lightgbm>=4.0")

# PyTorch (Colab 预装 CUDA 版，只需确认)
import torch
print(f"PyTorch {torch.__version__}  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print("\n依赖安装完成 ✓")

In [ ]:
# @title 克隆 EternityQuant
import os
REPO_URL = "https://github.com/Eternity231/EternityQuant.git"  # TODO: 替换为你的仓库地址
PROJECT_DIR = "/content/EternityQuant"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull

os.chdir(PROJECT_DIR)
!pip install -e . --quiet
print(f"工作目录: {os.getcwd()}")

## 2. 准备 qlib 数据

### 方案 A：上传本地打包的 `cn_data.tar.gz`（推荐，快）
在本地跑 `eq data a --start 2015-01-01 --universe all --workers 5` 下载全量 A 股数据，
然后用 `python -c "import tarfile, os; os.chdir('.qlib_data'); tarfile.open('../cn_data.tar.gz', 'w:gz').add('cn_data/')"` 打包上传。

### 方案 B：在 Colab 中在线拉取（慢，约 30 分钟）
用腾讯 API 直接拉取全量 A 股数据。

In [ ]:
# @title 挂载 Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# @title 方案 A：从 Google Drive 解压 cn_data.tar.gz
import os

# 如果 cn_data.tar.gz 在 Google Drive 中，从 Drive 复制
DRIVE_TAR = "/content/drive/MyDrive/qlib_data/cn_data.tar.gz"
if os.path.exists(DRIVE_TAR):
    !mkdir -p /content/.qlib_data
    !tar xzf "{DRIVE_TAR}" -C /content/.qlib_data/
    print(f"已从 Drive 解压: {DRIVE_TAR}")
elif os.path.exists("cn_data.tar.gz"):
    # 文件已上传到当前目录
    !mkdir -p /content/.qlib_data
    !tar xzf cn_data.tar.gz -C /content/.qlib_data/
    print("已从当前目录解压 cn_data.tar.gz")
else:
    print("未找到 cn_data.tar.gz，请上传或切换到方案 B")

In [ ]:
# @title 方案 B：在线拉取全量 A 股数据（腾讯 API）
import os
QLIB_DATA_DIR = "/content/.qlib_data/cn_data"
if not os.path.exists(QLIB_DATA_DIR):
    !mkdir -p /content/.qlib_data
    !cd /content/EternityQuant && python -c "
from eq.strategy.factors.ml_data_updater import update_qlib_data
result = update_qlib_data(start='2015-01-01', universe='all', workers=5, verbose=True)
print(f'数据更新完成: {result}')
"
else:
    print(f"qlib 数据已存在: {QLIB_DATA_DIR}")

# 验证
!ls /content/.qlib_data/cn_data/calendars/ 2>/dev/null || echo "日历文件未找到"
!echo "特征数: $(ls /content/.qlib_data/cn_data/features/ 2>/dev/null | wc -l)"

## 3. 训练模型

### 3.1 LightGBM (CPU 基线)

In [ ]:
# @title 训练 LightGBM
import sys, os
os.chdir("/content/EternityQuant")
sys.path.insert(0, ".")

from eq.strategy.factors.ml_workflow import train

result = train(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2020-08-31",
    valid_start="2020-09-01",
    valid_end="2020-09-25",
    horizon=5,
    algo="lightgbm",
    device="cpu",
    name="colab_lgbm_csi300_h5",
)
print(f"\n训练完成:")
print(f"  Model ID: {result['model_id']}")
print(f"  IC: {result['metrics']['ic']:.4f}")
print(f"  模型文件: {result['model_path']}")

### 3.2 MLP (CUDA)

In [ ]:
# @title 训练 MLP
import sys, os
os.chdir("/content/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2020-08-31",
    valid_start="2020-09-01",
    valid_end="2020-09-25",
    horizon=5,
    algo="mlp",
    device="cuda",
    name="colab_mlp_csi300_h5",
)
print(f"\n训练完成:")
print(f"  Model ID: {result['model_id']}")
print(f"  IC: {result['metrics']['ic']:.4f}")
print(f"  模型文件: {result['model_path']}")

### 3.3 GRU (CUDA) — 推荐，量化选股最佳

In [ ]:
# @title 训练 GRU (T4 优化参数)
import sys, os
os.chdir("/content/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

# T4 16GB 优化参数: batch=1024, hidden=128, layers=2
result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2020-08-31",
    valid_start="2020-09-01",
    valid_end="2020-09-25",
    horizon=5,
    algo="gru",
    device="cuda",
    hidden_size=128,
    num_layers=2,
    batch_size=1024,
    name="colab_gru_csi300_h5",
)
print(f"\n训练完成:")
print(f"  Model ID: {result['model_id']}")
print(f"  IC: {result['metrics']['ic']:.4f}")
print(f"  Epochs: {result['metrics']['epochs']}")
print(f"  模型文件: {result['model_path']}")

### 3.4 LSTM (CUDA)

In [ ]:
# @title 训练 LSTM (T4 优化参数)
import sys, os
os.chdir("/content/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

# T4 16GB 优化参数: batch=1024, hidden=128, layers=2
result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2020-08-31",
    valid_start="2020-09-01",
    valid_end="2020-09-25",
    horizon=5,
    algo="lstm",
    device="cuda",
    hidden_size=128,
    num_layers=2,
    batch_size=1024,
    name="colab_lstm_csi300_h5",
)
print(f"\n训练完成:")
print(f"  Model ID: {result['model_id']}")
print(f"  IC: {result['metrics']['ic']:.4f}")
print(f"  模型文件: {result['model_path']}")

### 3.5 DeepLOB (CUDA) — CNN+BiLSTM+Attention

[Zhang et al., 2019] 针对金融微观结构设计的专用架构。
卷积层提取空间特征，BiLSTM 建模时序，注意力机制加权。

In [ ]:
# @title 训练 DeepLOB (T4 优化参数)
import sys, os
os.chdir("/content/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

# T4 16GB 优化参数: batch=256, seq_len=120, hidden=64, dropout=0.3
result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2020-08-31",
    valid_start="2020-09-01",
    valid_end="2020-09-25",
    horizon=5,
    algo="deeplob",
    device="cuda",
    optimizer="adamw",
    loss="sharpe",
    batch=256,
    name="colab_deeplob_csi300_h5",
)
print(f"\n训练完成:")
print(f"  Model ID: {result['model_id']}")
print(f"  IC: {result['metrics']['ic']:.4f}")
print(f"  模型文件: {result['model_path']}")

### 3.6 TFT (CUDA) — Temporal Fusion Transformer

[Lim et al., 2019] Google 多时间跨度预测模型。
多头注意力+GRN+位置编码，目前中低频时序最先进模型之一。

In [ ]:
# @title 训练 TFT (T4 优化参数)
import sys, os
os.chdir("/content/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

# T4 16GB 优化参数: batch=128, hidden=128, heads=4, dropout=0.4
result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2020-08-31",
    valid_start="2020-09-01",
    valid_end="2020-09-25",
    horizon=5,
    algo="tft",
    device="cuda",
    optimizer="adamw",
    loss="sharpe",
    hidden=128,
    heads=4,
    batch=128,
    name="colab_tft_csi300_h5",
)
print(f"\n训练完成:")
print(f"  Model ID: {result['model_id']}")
print(f"  IC: {result['metrics']['ic']:.4f}")
print(f"  模型文件: {result['model_path']}")

### 3.7 高级优化器对比

| 优化器 | 核心思想 | 金融优势 |
|--------|---------|---------|
| SAM | 寻找平坦极小值 | 市场环境漂移时仍保持低损失 |
| Lookahead | 双权重：快权探索，慢权稳定 | 降低局部噪音带偏概率 |
| Lion | 只看梯度符号，忽略幅度 | 免疫闪崩等极端异常值 |

In [ ]:
# @title 训练 GRU + SAM 优化器 + 可微夏普比率
import sys, os
os.chdir("/content/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2020-08-31",
    valid_start="2020-09-01",
    valid_end="2020-09-25",
    horizon=5,
    algo="gru",
    device="cuda",
    optimizer="sam",     # Sharpness-Aware Minimization
    loss="sharpe",       # 可微夏普比率损失
    name="colab_gru_sam_csi300_h5",
)
print(f"\n训练完成: IC={result['metrics']['ic']:+.4f}")

In [ ]:
# @title 训练 GRU + Lion 优化器
import sys, os
os.chdir("/content/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2020-08-31",
    valid_start="2020-09-01",
    valid_end="2020-09-25",
    horizon=5,
    algo="gru",
    device="cuda",
    optimizer="lion",     # EvoLved Sign Momentum
    loss="sharpe",
    name="colab_gru_lion_csi300_h5",
)
print(f"\n训练完成: IC={result['metrics']['ic']:+.4f}")

### 3.8 对抗训练 + 特征正交化

FGSM 对抗训练 + 特征正交化去 Beta，提升模型在实盘异常行情中的鲁棒性。

In [ ]:
# @title 训练 TFT + 对抗训练 + 特征正交化
import sys, os
os.chdir("/content/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2020-08-31",
    valid_start="2020-09-01",
    valid_end="2020-09-25",
    horizon=5,
    algo="tft",
    device="cuda",
    optimizer="adamw",
    loss="sharpe",
    adversarial=True,       # FGSM 对抗训练
    orthogonalize=True,     # 特征正交化去 Beta
    name="colab_tft_adv_orth_csi300_h5",
)
print(f"\n训练完成: IC={result['metrics']['ic']:+.4f}")

### 3.9 多卡并行训练（多 GPU 服务器专用）

Colab 只有 1 张 T4，但如果租用多 GPU 云服务器（如 A100×4、4090×2），
可以用 `--gpus` 参数启用 `nn.DataParallel` 自动并行：

```bash
# 双卡训练 TFT
eq ml train csi300 5 --algo tft --device cuda --gpus "0,1"

# 四卡训练 DeepLOB
eq ml train csi300 5 --algo deeplob --device cuda --gpus "0,1,2,3"

# 港股训练双卡
eq hk train --top 73 --device cuda --gpus "0,1"
```

### 3.10 超参搜索（自动找最佳 LSTM/GRU 参数）

In [ ]:
# @title GRU 超参网格搜索
import sys, os
os.chdir("/content/EternityQuant")

from eq.strategy.factors.ml_workflow import search_lstm

results = search_lstm(
    universe="csi300",
    horizon=5,
    train_start="2015-01-01",
    train_end="2020-08-31",
    valid_start="2020-09-01",
    valid_end="2020-09-25",
    device="cuda",
    fast=True,        # 快速模式：每组合 max_steps=50
    auto_train=True,  # 搜索完自动用最佳参数全量训练
    algo="gru",      # 可选 "lstm" 或 "gru"
)

# 打印 Top3 结果
print(f"\n搜索完成 {len(results)} 组合")
for i, r in enumerate(results[:3]):
    print(f"  #{i+1}: hidden={r['hidden_size']} layers={r['num_layers']} "
          f"lr={r['lr']} batch={r['batch_size']}  IC={r['ic']:+.4f}")

## 4. 导出模型回本地

训练好的模型文件在 `/content/EternityQuant/.eternityquant/ml_models/` 目录下。
将其下载到本地，然后用 `eq ml register` 和 `eq ml activate` 导入。

In [ ]:
# @title 列出所有训练好的模型
import os
models_dir = "/content/EternityQuant/.eternityquant/ml_models"
if os.path.exists(models_dir):
    for f in sorted(os.listdir(models_dir)):
        size = os.path.getsize(os.path.join(models_dir, f))
        print(f"  {f:45s}  {size/1024:.1f} KB")
else:
    print("未找到模型文件")

In [ ]:
# @title 打包下载所有模型到本地
import os
models_dir = "/content/EternityQuant/.eternityquant/ml_models"
if os.path.exists(models_dir) and os.listdir(models_dir):
    !tar czf /content/eternityquant_models.tar.gz -C {models_dir} .
    from google.colab import files
    files.download("/content/eternityquant_models.tar.gz")
else:
    print("没有模型文件可下载")

## 5. 回本地导入模型

```bash
# 解压模型文件
tar xzf eternityquant_models.tar.gz -C ~/.eternityquant/ml_models/

# 登记模型
eq ml register --name "colab_gru_csi300_h5" --universe csi300 \
    --features "Alpha158" --algo gru --horizon 5 \
    --train-period "2015-01-01~2020-08-31" \
    --model-path ~/.eternityquant/ml_models/torch_gru_csi300_5d.pkl \
    --metrics '{"ic": 0.15}' \
    --notes "Colab T4 GPU 训练"

# 激活模型
eq ml activate <model_id>

# 批量预测今日 Top 10
eq ml predict-batch <model_id> --top 10
```

---

## 附：Colab GPU 配置信息

In [ ]:
# @title GPU 信息
import torch, psutil, platform
print(f"系统: {platform.platform()}")
print(f"CPU: {psutil.cpu_count()} 核")
print(f"内存: {psutil.virtual_memory().total / 1e9:.1f} GB")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA 版本: {torch.version.cuda}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("GPU: 不可用（将使用 CPU）")